In [4]:
import os
from dotenv import load_dotenv
load_dotenv()
redis_url=os.getenv('REDIS_URL')
import redis
db=redis.from_url(redis_url)


In [7]:
print(db.get('test'))

b'abc'


In [9]:
%cd jobcrawl
from jobcrawl.cache import RedisCacheStorage
from scrapy.settings import Settings
from jobcrawl import settings as s
cfg = Settings()
cfg.setmodule(s)
storage = RedisCacheStorage(cfg)
class FakeSpider: 
    name = 'test'
    crawler = None
storage.open_spider(FakeSpider())
print('Redis ping:', storage.db.ping())
storage.close_spider(FakeSpider())
print('OK')

/Users/lebaminhphuc/Desktop/2025.2/các môn đại học/đồ án/build_code_new/jobcrawl


ModuleNotFoundError: No module named 'jobcrawl.cache'

In [10]:
from scrapy.settings import Settings
from jobcrawl.cache import RedisCacheStorage
from jobcrawl import settings as s

cfg = Settings()
cfg.setmodule(s)
storage = RedisCacheStorage(cfg)

class FakeSpider:
    name = 'test'
    crawler = None

storage.open_spider(FakeSpider())
print('Redis ping:', storage.db.ping())
storage.close_spider(FakeSpider())
print('OK')

ModuleNotFoundError: No module named 'jobcrawl.cache'

In [8]:
import sys, os
sys.path.insert(0, "/Users/lebaminhphuc/Desktop/2025.2/các môn đại học/đồ án/build_code_new/jobcrawl")
os.chdir("/Users/lebaminhphuc/Desktop/2025.2/các môn đại học/đồ án/build_code_new/jobcrawl")

import hashlib
import redis
from scrapy.utils.project import get_project_settings
from jobcrawl.cache import make_key

settings = get_project_settings()
r = redis.from_url(settings.get("HTTPCACHE_REDIS_URL"))

import gzip, zlib

def _maybe_decompress(body: bytes) -> tuple[bytes, str]:
    """Trả về (decoded_body, encoding_detected)."""
    if body[:2] == b"\x1f\x8b":
        return gzip.decompress(body), "gzip"
    if body[:2] in (b"\x78\x9c", b"\x78\x01", b"\x78\xda"):
        return zlib.decompress(body), "deflate"
    # brotli (Scrapy hỗ trợ nếu cài brotli)
    if body[:3] == b"\xce\xb2\xcf":   # không phải magic chuẩn, brotli không có magic
        pass
    try:
        import brotli
        return brotli.decompress(body), "brotli"
    except Exception:
        pass
    return body, "none"

def peek(key_or_url: str, preview_bytes: int = 500):
    key = make_key(key_or_url) if not key_or_url.startswith("jobcrawl:cache:") else key_or_url
    body = r.get(key)
    if body is None:
        return {"key": key, "exists": False}

    decoded, enc = _maybe_decompress(body)
    return {
        "key": key,
        "exists": True,
        "size_compressed": len(body),
        "size_decoded": len(decoded),
        "encoding": enc,
        "ttl_seconds": r.ttl(key),
        "preview": decoded[:preview_bytes].decode("utf-8", errors="replace"),
    }


def peek_full(key_or_url: str) -> bytes:
    """Trả về toàn bộ body (bytes). None nếu không có."""
    key = make_key(key_or_url) if not key_or_url.startswith("jobcrawl:cache:") else key_or_url
    return r.get(key)


def list_keys(pattern: str = "jobcrawl:cache:*", limit: int = 20):
    """Liệt kê các key đang có trong Redis cache."""
    out = []
    for k in r.scan_iter(match=pattern, count=100):
        out.append(k.decode())
        if len(out) >= limit:
            break
    return out


In [9]:
len(list_keys())

10

In [10]:
peek('jobcrawl:cache:052b1ff073d3a93960861f20175f133a')['preview']

'\n<!DOCTYPE html>\n<html lang="vi">\n\n<head>\n    <meta charset="utf-8">\n    <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1">\n    <link rel="icon" href="https://static.topcv.vn/v4/image/favicon.ico" type="image/x-icon">\n    <meta property="fb:app_id" content="1478418029113221" />\n    <meta name="csrf-token" content="XzCtQBxKlpnOMSS7poltnRigC0te0w7YhBo0HrKA">\n    <meta name="google-site-verificat'